# 07 Dengue sa susunod na buwan: forecasting gamit ang TimesFM — notebook

Kailangan ni Doc Mon ng mga numero para sa Local Health Board: ilang kaso ng dengue ang dapat paghandaan ng Santa Luz sa susunod na walong linggo? Sa notebook na ito, gagawin mo ang mga sumusunod:

1. tingnan ang walong taon ng lingguhang bilang ng dengue (`data/synthetic/ph/santa_luz_dengue_weekly.csv`, **synthetic**);
2. bumuo muna ng dalawang simpleng **baseline** — sila ang pamantayang dapat lampasan ng anumang model;
3. patakbuhin ang **TimesFM 3.0**, isang zero-shot na time-series foundation model, sa isang patas na **rolling-origin backtest**;
4. tingnan ang ginagawa nito kapag umarangkada ang outbreak, at kung nakakatulong ang ulan;
5. gawin ang totoong forecast mula 2026-W39 — may uncertainty band — at pagpasyahan kung gaano ito mapagkakatiwalaan.

> **Dalawang daan, parehong numero.** Sa Google Colab (o sa anumang computer na may naka-install na `timesfm`), talagang tinatawag ng mga cell sa ibaba ang TimesFM 3.0. Kapag binubuo ang website na ito, hindi dina-download ang 1.3 GB na model: inuulit ng `epi_learning.tsfm` ang mga forecast na na-save mula sa isang totoong run ng TimesFM 3.0 sa parehong notebook na ito (`data/tsfm_cache/ph07_dengue.json`). Ipinapakita ng bawat forecasting cell kung aling daan ang tinahak nito.

In [ ]:
# Google Colab setup -- skip this cell if running locally
import sys
import os
if 'google.colab' in sys.modules:
    !git clone https://github.com/ancientsky/python4epi.git /content/python4epi 2>/dev/null || true
    os.chdir('/content/python4epi')
    sys.path.insert(0, '/content/python4epi/src')
    !pip install -q -e .

In [ ]:
# Dagdag na hakbang sa Colab: i-install ang TimesFM 3.0 (mga 2 minuto; mamaya pa nada-download ang weights, ~1.3 GB)
# Sa sarili mong computer na may uv:  uv sync --group timesfm
import sys
if 'google.colab' in sys.modules:
    !pip install -q "timesfm[torch]>=3.0.2"

## Hakbang 1 — Ang lingguhang series ng dengue

Isang row bawat **morbidity week** (ISO week, Lunes hanggang Linggo), mula 2019-W01 hanggang 2026-W39, na may bilang ng kaso ng dengue ayon sa linggo ng onset at ang ulan sa weather station ng lungsod. Itinatago natin ang mga kaso bilang simpleng NumPy array na `y`: listahan ng mga numerong sunod-sunod ayon sa oras ang kailangan ng mga forecasting model.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from epi_learning import tsfm

weekly = pd.read_csv("data/synthetic/ph/santa_luz_dengue_weekly.csv", parse_dates=["week_start"])
y = weekly["cases"].to_numpy(dtype=float)          # ang target na series
rain = weekly["rainfall_mm"].to_numpy(dtype=float)  # posibleng pantulong na series (covariate)
week_label = weekly["morbidity_week"].to_numpy()
row_of = {label: i for i, label in enumerate(week_label)}  # "2026-W31" -> numero ng row

print(f"{len(weekly)} linggo: {week_label[0]} hanggang {week_label[-1]}")
print(weekly.tail(6)[["morbidity_week", "cases", "rainfall_mm"]].to_string(index=False))

fig, ax = plt.subplots(figsize=(11, 3.8))
ax.plot(weekly["week_start"], y, color="#0038A8", linewidth=1.4)
ax.axvspan(pd.Timestamp("2026-07-13"), weekly["week_start"].iloc[-1], color="#CE1126", alpha=0.12,
           label="Pagkatapos ng Bagyong Amihan")
ax.set_ylabel("Kaso ng dengue bawat linggo")
ax.set_title("Santa Luz City (synthetic): lingguhang kaso ng dengue, 2019-W01 hanggang 2026-W39")
ax.legend(loc="upper left")
plt.tight_layout()
plt.show()

## Hakbang 2 — Baseline muna

Bago ang anumang matalinong model, hinihingi ni Maya kay Tala ang dalawang pinakasimpleng forecast na kayang gawin sa papel ng isang pagod na DSO:

> **Tanungin si Tala:** *May NumPy array akong `y` ng lingguhang bilang ng dengue, sunod-sunod ayon sa oras (gawa-gawang halimbawa: `[5, 9, 11, 4, ...]`, mahigit 52 linggo). Para sa forecast na ginawa sa row `o` (`y[:o]` lang ang alam natin), gusto ko ang susunod na `H = 8` linggo mula sa dalawang baseline: (1) "last value" — ulitin ang `y[o-1]` nang walong beses; (2) "seasonal naive" — kopyahin ang parehong walong linggo noong nakaraang taon, `y[o-52 : o-52+8]`. Gawin ito para sa isang listahan ng origin at ibigay ang mean absolute error (MAE) laban sa totoong value. Magdagdag ng check na kaya kong gawin nang mano-mano.*

Ang sagot ni Tala ay ang cell sa ibaba. Ang mga **origin** ay tuwing ikalawang linggo mula 2023-W01 hanggang 2026-W31: 94 na forecast, bawat isa 8 linggo ang haba, at bawat isa ay gumagamit lang ng nakaraan. Ito ang **rolling-origin backtest**: nagkukunwari tayong nakatayo sa bawat origin, gumagawa ng forecast, at saka lang tinitingnan ang totoong nangyari.

In [ ]:
H = 8                                             # forecast horizon: 8 linggo
first, last = row_of["2023-W01"], row_of["2026-W39"] + 1
origins = list(range(first, last - H + 1, 2))     # tuwing ika-2 linggo; hindi lalampas sa 2026-W39 ang dulo ng bawat forecast
actual = np.stack([y[o:o + H] for o in origins])  # shape (94, 8): ang totoong nangyari

last_value = np.stack([np.repeat(y[o - 1], H) for o in origins])
seasonal_naive = np.stack([y[o - 52:o - 52 + H] for o in origins])  # walang ISO week 53 pagkatapos ng 2020


def mae(pred, truth=actual):
    # Mean absolute error: ang karaniwang laki ng sablay, sa kaso kada linggo
    return float(np.mean(np.abs(np.asarray(pred) - np.asarray(truth))))


print(f"{len(origins)} origin, nagsisimula ang unang forecast sa {week_label[origins[0]]}, "
      f"ang huli sa {week_label[origins[-1]]}")
print(f"MAE last value:     {mae(last_value):.2f} kaso/linggo")
print(f"MAE seasonal naive: {mae(seasonal_naive):.2f} kaso/linggo")

**I-check natin.** Huwag kailanman magtiwala sa backtest na hindi mo pa nasusundot. Kumuha tayo ng isang origin at suriin nang mano-mano na (a) talagang inuulit ng "last value" ang linggo bago ang origin, (b) talagang kinokopya ng "seasonal naive" ang parehong mga linggo isang taon bago, at (c) walang forecast na gumagamit ng linggo sa o pagkatapos ng sarili nitong origin.

In [ ]:
k = 10                                   # kahit anong origin
o = origins[k]
print("Forecast na ginawa pagkatapos ng linggo:", week_label[o - 1], "| unang linggo ng forecast:", week_label[o])
print("Inuulit ng last value ang", y[o - 1], "->", last_value[k][:3], "...")
print("Kinokopya ng seasonal naive ang", week_label[o - 52], "...", week_label[o - 52 + H - 1], "->", seasonal_naive[k])

assert np.all(last_value[k] == y[o - 1])
assert week_label[o - 52][5:] == week_label[o][5:], "parehong ISO week isang taon bago"
assert np.array_equal(actual[k], y[o:o + H])
assert all(len(c) == o for c, o in zip(tsfm.rolling_origin_contexts(y, origins), origins)), "dapat huminto sa origin ang history"
print("Pasado lahat ng check.")

## Hakbang 3 — I-load ang TimesFM 3.0

Ang TimesFM ay isang **time-series foundation model**: minsan itong sinanay ng Google sa napakalaking koleksiyon ng time series, at ginagamit natin ito nang **zero-shot** — walang training sa datos natin, "heto ang history, ibigay mo ang susunod na 8 linggo" lang. Nagbabalik ito ng median na forecast at siyam na **quantile** (0.1 … 0.9); ang 0.1 at 0.9 na quantile ang hangganan ng **80% prediction interval**.

Tatlong linya lang ang opisyal na API (`ModelConfig`, `TimesFM3Evaluator`, `predict_batch`). Kung hindi naka-install ang `timesfm`, lilipat ang cell na ito sa cache at sasabihin nito iyon.

In [ ]:
CACHE = tsfm.cache_path("ph07_dengue")   # data/tsfm_cache/ph07_dengue.json
MODE = tsfm.resolve_mode()               # "model" kung naka-install ang timesfm, kung hindi ay "cache"

if MODE == "cache":
    model = None
    print("Daan = CACHE: hindi pinapatakbo ang TimesFM sa session na ito; inuulit ang mga forecast mula sa isang totoong run.")
    print(tsfm.describe_cache(CACHE))
else:
    from timesfm3 import ModelConfig, TimesFM3Evaluator

    config = ModelConfig(
        checkpoint_path="google/timesfm-3.0-pytorch",  # weights sa Hugging Face (~1.3 GB, sa unang run lang)
        per_core_batch_size=32,                        # ilang series bawat forward pass
        device=tsfm.default_device(),                  # "cuda" kung GPU runtime, kung hindi ay "cpu"
    )
    model = TimesFM3Evaluator(config)
    print("Daan = MODEL: naka-load ang TimesFM 3.0 sa", tsfm.default_device())

## Hakbang 4 — Parehong backtest, gamit ang TimesFM

Eksaktong parehong 94 na origin, parehong 8-linggong horizon, parehong MAE. Ibinabalik ng `tsfm.rolling_origin_contexts(y, origins)` ang `y[:o]` para sa bawat origin — ang history na pinapayagang makita ng model. Kapag may model, tinatawag lang ng `tsfm.forecast_batch` ang `model.predict_batch(contexts, horizon=8, return_quantiles=True)`; sa CPU, ilang minuto ito.

Hiwalay nating sini-score ang dalawang panahon: ang **karaniwang mga taon 2023–2025** at ang **2026**, ang taon ng bagyo. Matatakpan ng iisang average ang pagkakaiba.

In [ ]:
contexts = tsfm.rolling_origin_contexts(y, origins)
bt = tsfm.forecast_batch(model, contexts, H, cache_key="backtest_cases", cache_file=CACHE)
tfm = bt.forecast                                   # mga median na forecast, shape (94, 8)
lo, hi = bt.quantile(0.1), bt.quantile(0.9)         # 80% prediction interval

year_of_end = weekly["iso_year"].to_numpy()[[o + H - 1 for o in origins]]
periods = {"2023-2025": year_of_end <= 2025, "2026": year_of_end == 2026}

rows = []
for period, sel in periods.items():
    for name, pred in [("Last value", last_value), ("Seasonal naive", seasonal_naive), ("TimesFM 3.0", tfm)]:
        err = np.abs(pred[sel] - actual[sel])
        rows.append({"panahon": period, "paraan": name, "origin": int(sel.sum()),
                     "MAE": err.mean(), "MAE week 1": err[:, 0].mean(), "MAE week 8": err[:, -1].mean()})
    rows.append({"panahon": period, "paraan": "Saklaw ng 80% interval ng TimesFM",
                 "origin": int(sel.sum()), "MAE": tsfm.coverage(actual[sel], lo[sel], hi[sel])})
score = pd.DataFrame(rows)
print(score.round(2).to_string(index=False))

### Pagbasa sa scoreboard

- **Karaniwang mga taon (2023–2025, 75 forecast):** **3.50** kaso kada linggo ang MAE ng TimesFM, laban sa **4.31** ng seasonal naive at **4.76** ng last value. Nasa loob ng 80% interval nito ang totoong value **79%** ng pagkakataon — halos iyon ang dapat gawin ng isang tapat na 80% band. Kahit walang anumang training, kapaki-pakinabang na forecaster ito.
- **2026 (19 forecast na natatapos sa 2026):** **5.32** ang MAE ng TimesFM, **5.21** ang sa seasonal naive. Dito, **hindi mas magaling ang model kaysa sa pagkopya ng nakaraang taon**. Ito ang pinakamahusay para sa susunod na linggo (MAE 1.97 sa week 1) pero napag-iiwanan pagdating ng week 8 (10.08 laban sa 8.68 ng seasonal naive). **72%** lang ang nasaklaw ng band.
- Aral: natakpan sana ito ng iisang kabuuang score. Laging i-score nang hiwalay ang di-pangkaraniwang panahon — iyon ang panahong mahalaga sa Local Health Board.

## Nagkamali si Tala: ang forecast na nakakita na ng sagot

Humingi si Maya kay Tala ng "mas magandang baseline: ang average ng parehong linggo sa lahat ng taon". Napakaganda ng score ng code ni Tala — MAE na mas mababa nang malaki sa lahat ng nasa itaas. Sampung segundong tumingin si Doc Mon at nagtanong: *"Aling mga taon ang pumasok sa average na iyan?"* Lahat — **kasama ang mismong mga linggong hinuhulaan**. Iyan ang **leakage**: ipinakita sa forecast ang sagot. Inuulit ng cell sa ibaba ang bersiyon ni Tala, tapos ang tapat na bersiyong ina-average lang ang mga taon **bago** ang origin.

Ang ikalawa at mas tahimik na paraan para lokohin ang sarili ay ang pag-uulat lang ng **pinakamagandang** linggo o origin. Ipi-print din natin iyon, katabi ng tapat na average.

In [ ]:
iso_week = weekly["iso_week"].to_numpy()
iso_year = weekly["iso_year"].to_numpy()

# Bersiyon ni Tala (MALI): average ng parehong ISO week sa LAHAT ng taon, kasama ang mga test year
all_years_mean = weekly.groupby("iso_week")["cases"].mean()
leaky = np.stack([all_years_mean.loc[iso_week[o:o + H]].to_numpy() for o in origins])

# Tapat na bersiyon: ang tatlong buong taon lang bago ang taon ng origin
honest = []
for o in origins:
    past = weekly[(iso_year >= iso_year[o] - 3) & (iso_year < iso_year[o])]
    honest.append(past.groupby("iso_week")["cases"].mean().reindex(iso_week[o:o + H]).to_numpy())
honest = np.stack(honest)

print(f"Same-week average, lahat ng taon (MAY LEAKAGE): MAE {mae(leaky):.2f}  <- mukhang maganda, walang saysay")
print(f"Same-week average, 3 nakaraang taon lang:      MAE {mae(honest):.2f}")
print(f"TimesFM 3.0 (nakaraan lang ang nakita):        MAE {mae(tfm):.2f}")

per_origin = np.abs(tfm - actual).mean(axis=1)
print(f"\nCherry-picking: MAE {per_origin.min():.2f} ang pinakamagandang origin ng TimesFM "
      f"({week_label[origins[int(per_origin.argmin())]]}), {per_origin.max():.2f} ang pinakamasama "
      f"({week_label[origins[int(per_origin.argmax())]]}); {per_origin.mean():.2f} ang tapat na average.")

## Hakbang 5 — Ang mahirap na sandali: umaarangkada ang outbreak

Ang pinakamahalagang mga linggo ay iyong lumiliko ang curve. Ibinibigay natin sa TimesFM ang history hanggang 2026-W30, W31, W32 at W34, hinihingi ang susunod na 8 linggo bawat pagkakataon, at iginuguhit ang mga forecast sa ibabaw ng totoong nangyari. (Ang 2026-W32 ang linggong unang lumampas ang dengue sa epidemic threshold sa [kabanata 06](../06_signals_and_thresholds.md).)

In [ ]:
takeoff_weeks = ["2026-W30", "2026-W31", "2026-W32", "2026-W34"]
takeoff_origins = [row_of[w] + 1 for w in takeoff_weeks]       # kasama sa history ang linggong iyon
take = tsfm.forecast_batch(model, tsfm.rolling_origin_contexts(y, takeoff_origins), H,
                           cache_key="takeoff", cache_file=CACHE)

fig, axes = plt.subplots(1, 4, figsize=(13, 3.6), sharey=True)
show = slice(row_of["2026-W20"], len(y))
for ax, k, w in zip(axes, range(4), takeoff_weeks):
    o = takeoff_origins[k]
    steps = np.arange(o, o + H)
    ax.bar(np.arange(len(y))[show], y[show], color="#9bbaff", label="Naiulat na kaso")
    ax.plot(steps, take.forecast[k], color="#CE1126", marker="o", ms=3, label="Median ng TimesFM")
    ax.fill_between(steps, take.quantile(0.1)[k], take.quantile(0.9)[k], color="#CE1126", alpha=0.18,
                    label="80% interval")
    ax.axvline(o - 0.5, color="#1c1b29", linestyle="--", linewidth=1)
    ax.set_title(f"Alam ang datos hanggang {w}", fontsize=10)
    ax.set_xticks([row_of["2026-W24"], row_of["2026-W32"], row_of["2026-W39"]], ["W24", "W32", "W39"])
    known = min(H, len(y) - o)
    inside = tsfm.coverage(y[o:o + known], take.quantile(0.1)[k][:known], take.quantile(0.9)[k][:known])
    print(f"Hanggang {w}: median sa susunod na 4 na linggo {np.round(take.forecast[k][:4]).astype(int)}, "
          f"totoo {y[o:o + 4].astype(int)}, totoong nasa loob ng 80% band: {inside:.0%} ng {known} linggo")
axes[0].set_ylabel("Kaso ng dengue bawat linggo")
axes[0].legend(loc="upper left", fontsize=8)
plt.tight_layout()
plt.show()

## Hakbang 6 — Makakatulong ba ang ulan?

Sa synthetic na series na ito, **nauuna** ang ulan sa mga kaso: pinakamataas ang correlation kapag inusog ang ulan nang 7 linggo (tingnan ang README ng datos). Tumatanggap ang TimesFM 3.0 ng mga **covariate**. Ibinibigay ng `past_only_covariates` ang ulan hanggang sa origin — nasa history na ang ulan ng Bagyong Amihan (2026-W29) kapag nag-forecast tayo mula W31. Ginagamit ba ito ng model? Parehong backtest, parehong take-off origin, ngayon kasama ang ulan.

In [ ]:
rain_past = [rain[None, :o] for o in origins]            # isang (1, T) na array bawat origin
bt_rain = tsfm.forecast_batch(model, contexts, H, cache_key="backtest_rain", cache_file=CACHE,
                              past_only_covariates=rain_past)
take_rain = tsfm.forecast_batch(model, tsfm.rolling_origin_contexts(y, takeoff_origins), H,
                                cache_key="takeoff_rain", cache_file=CACHE,
                                past_only_covariates=[rain[None, :o] for o in takeoff_origins])

for period, sel in periods.items():
    print(f"{period}: MAE, kaso lang {mae(tfm[sel], actual[sel]):.2f} | "
          f"kasama ang ulan {mae(bt_rain.forecast[sel], actual[sel]):.2f}")
k = takeoff_weeks.index("2026-W31")
print("\nMula 2026-W31, susunod na 8 linggo")
print("  totoo           ", y[takeoff_origins[k]:takeoff_origins[k] + H].astype(int))
print("  kaso lang       ", np.round(take.forecast[k]).astype(int))
print("  kasama ang ulan ", np.round(take_rain.forecast[k]).astype(int))

## Hakbang 7 — Ang forecast na hiningi ni Doc Mon: 2026-W40 hanggang W47

Ngayon, ang totoong forecast: lahat ng history hanggang 2026-W39, ang susunod na 8 linggo, kasama ang 80% interval. Pero tandaan ang [kabanata 06](../06_signals_and_thresholds.md): **kulang pa ang W39** — sa freeze noong 30 Setyembre, inuulat pa ang mga kasong nag-onset noong linggong iyon. Kaya pinapatakbo rin natin ang model sa history **nang wala ang W39** at ikinukumpara. Kung malaki ang pagkakaiba ng dalawa, ang pinakabagong mga linggo ang nagmamaneho sa forecast at dapat mo itong sabihin.

In [ ]:
if model is not None:
    # Ang totoong tawag sa TimesFM 3.0, minsan, nang walang helper:
    out = next(model.predict_batch([y.astype(np.float32)], horizon=H, return_quantiles=True))
    print("predict_batch -> forecast", out.forecast.shape, "quantiles", out.quantiles.shape)

now = tsfm.forecast_batch(model, [y, y[:-1]], H, cache_key="forecast_now", cache_file=CACHE)
future_weeks = [f"2026-W{w}" for w in range(40, 48)]
table = pd.DataFrame({
    "linggo": future_weeks,
    "mababa (q0.1)": now.quantile(0.1)[0],
    "median": now.forecast[0],
    "mataas (q0.9)": now.quantile(0.9)[0],
    "median kung wala ang W39": now.forecast[1][1:].tolist() + [np.nan],
}).round(0)
print(table.to_string(index=False))
print(f"\nMula sa datos hanggang W38, {now.forecast[1][0]:.0f} kaso ang inaasahan ng model sa W39; "
      f"{y[-1]:.0f} ang naiulat hanggang sa freeze.")
print(f"Kabuuan ng mga lingguhang median, W40-W47: {now.forecast[0].sum():.0f} kaso")
print(f"Parehong 8 linggo noong nakaraang taon (2025-W40 hanggang W47): {y[len(y) - 52:len(y) - 52 + H].sum():.0f} kaso")

fig, ax = plt.subplots(figsize=(10, 3.8))
past = np.arange(row_of["2026-W20"], len(y))
ax.bar(past, y[past], color="#9bbaff", label="Naiulat na kaso (kulang pa ang W39)")
steps = np.arange(len(y), len(y) + H)
ax.plot(steps, now.forecast[0], color="#CE1126", marker="o", ms=3, label="Median ng TimesFM")
ax.fill_between(steps, now.quantile(0.1)[0], now.quantile(0.9)[0], color="#CE1126", alpha=0.18,
                label="80% prediction interval")
ax.plot(np.arange(len(y) - 1, len(y) - 1 + H), now.forecast[1], color="#2f7d32", linestyle="--",
        label="Median kung hindi isasama ang W39")
ax.set_xticks([row_of["2026-W24"], row_of["2026-W32"], row_of["2026-W39"], len(y) + 7],
              ["W24", "W32", "W39", "W47"])
ax.set_ylabel("Kaso ng dengue bawat linggo")
ax.set_title("Forecast mula 2026-W39 (synthetic na datos)")
ax.legend(loc="upper left", fontsize=8)
plt.tight_layout()
plt.show()

## Ikaw naman

Ginagamit ng mga ito ang mga array na hawak mo na — walang bagong tawag sa model, kaya gumagana ang mga ito sa website at sa Colab.

1. **Aling horizon ang pinakamahirap?** I-print ang MAE ng bawat paraan para sa bawat horizon week 1…8 (pahiwatig: walong numero ang ibinibigay ng `np.abs(tfm - actual).mean(axis=0)`). Saan nawawala ang lamang ng TimesFM sa seasonal naive noong 2026?
2. **Tapat ba ang 80% band?** Para sa 2023–2025, bilangin ang mga linggong nasa **itaas** ng band ang totoong value at ang nasa **ibaba** nito (pahiwatig: `(actual > hi)[sel].mean()`). Mga 10% ang nalalampasan ng isang tapat na band sa bawat panig.
3. **Plano, hindi hula.** Gamit ang `table`, sumulat ng isang pangungusap para sa Local Health Board na nagbibigay ng range, hindi iisang numero, at nagbabanggit ng isang bagay na magpapabago sa iyo nito.

In [ ]:
# 1. MAE ayon sa horizon week (panimula: TimesFM, mga origin ng 2026)
sel = periods["2026"]
print("TimesFM, 2026, MAE ayon sa linggong susunod:", np.abs(tfm[sel] - actual[sel]).mean(axis=0).round(1))
# ... ngayon, gawin din ito sa seasonal_naive at ikumpara

# 2. Mga sablay ng band (panimula)
sel = periods["2023-2025"]
print("Bahagi ng totoong value na nasa itaas ng band, 2023-2025:", round(float((actual > hi)[sel].mean()), 2))
# ... at sa ibaba?

## Bago mo ito gamitin sa trabaho: ang lisensiya

- Apache-2.0 ang **code** ng TimesFM, pero ang **weights ng TimesFM 3.0** (`google/timesfm-3.0-pytorch`) ay nasa ilalim ng **TimesFM Non-Commercial License v1.0**: para sa research, evaluation at pagtuturo, **bawal sa commercial at production na gamit**. Basahin mo mismo: [lisensiya ng TimesFM 3.0 sa Hugging Face](https://huggingface.co/google/timesfm-3.0-pytorch/blob/main/LICENSE).
- Apache-2.0 ang weights ng **TimesFM 2.5**. Nag-aalok ang Google ng commercial at production na gamit ng TimesFM sa pamamagitan ng Google Cloud.
- Ang ibig sabihin nito para sa isang health office ng gobyerno: pasok ang pag-aaral, pag-backtest sa lumang datos at pag-explore — gaya ng notebook na ito. Hindi pasok ang paggamit ng mga forecast ng 3.0 sa isang regular na early-warning system, sa pampublikong bulletin o sa regular na operational na desisyon. **Suriin ang lisensiya (at ang mga patakaran ng opisina mo) bago ang anumang operational na gamit.** Hindi ito legal advice.
- Ang mga naka-cache na forecast sa `data/tsfm_cache/ph07_dengue.json` ay mga output ng model na ginawa sa ilalim ng lisensiyang iyon para sa non-commercial na materyal pampagtuturong ito.

**Ang natutuhan mo:** talunin muna ang mga baseline sa isang patas na backtest bago maniwala sa isang model; i-score nang hiwalay ang mahihirap na linggo; huwag kailanman hayaang makita ng forecast ang sagot; at hindi nakikita ng model ang bagyo, ang bagong serotype, o ang mga kasong hindi pa naiuulat.